In [3]:
import sys
import os
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from backend.pipeline import (
    process_vocab_word
)

import json
import requests
from backend.constants import (
    ANKI_CONNECT_URL    
)

In [4]:
ANKI_CONNECT_URL

'http://localhost:8765'

In [5]:
def request(action, **params):
    return {'action': action, 'params': params, 'version': 6}

def invoke(action, **params):
    payload = request(action, **params)
    response = requests.post(ANKI_CONNECT_URL, json=payload)
    
    if not response.ok:
        raise Exception(f"Request failed with status code {response.status_code}: {response.text}")
    
    response_json = response.json()
    
    if 'error' not in response_json or 'result' not in response_json:
        raise Exception('Invalid response structure')
    if response_json['error'] is not None:
        raise Exception(response_json['error'])
    
    return response_json['result']

In [6]:
import base64

def store_audio_file(filename):
    abs_path = os.path.abspath(filename)  # Get absolute path of the file
    with open(abs_path, "rb") as f:
        audio_data = base64.b64encode(f.read()).decode("utf-8")  # Encode to Base64 and decode to a string
    
    print(abs_path)
    
    response = invoke("storeMediaFile", filename=filename, path=abs_path)
    print(f"Stored {filename}: {response}")  # Print Anki Connect response

In [9]:
invoke('modelNames')

['01 BASIC',
 '5000 French Words 2.0 (E to F)',
 '5000 French Words 2.0 (E to F) C',
 '5000 French Words 2.0 (F to E)',
 '5000 French Words 2.0 (F to E) C',
 'Arabic Abjad',
 'Basic',
 'Basic (and reversed card)',
 'Basic (optional reversed card)',
 'Basic (type in the answer)',
 'Basic+',
 'Basic-09604',
 'Cloze',
 'Cloze+',
 'Cloze++',
 'Core Japanese Vocabulary Extended',
 'French aspirated h',
 'French Ear Training',
 'French IPA deck',
 'French irregular pronunciation',
 'French phonology',
 'French sentences Read Training',
 'French sentences Speak Training',
 'French Verbs',
 'French vowels comparison',
 'Image Occlusion',
 'Intro card',
 'Japanese-75658',
 'Kanji Radical English',
 'Memrise - 8000+ Most Common Swedish Words - Part 1 (of four) - Swedish',
 'Memrise - 8000+ Most Common Swedish Words - Part 2 (of four) - Swedish',
 'Memrise - 8000+ Most Common Swedish Words - Part 3 (of four) - Swedish',
 'Memrise - 8000+ Most Common Swedish Words - Part 4 (of four) - Swedish',
 '

In [7]:
invoke('deckNames')

['* Navajo',
 '5000 Most Common French Words',
 '5000 Most Common French Words::[1] Main Course',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio::I) French to English (Start here)',
 '5000 Most Common French Words::[1] Main Course::[a] Option 1: Parisian French Audio::II) English to French',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio::I) French to English',
 '5000 Most Common French Words::[1] Main Course::[b] Option 2: Canadian French Audio::II) English to French',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training::[a] Option 1: Most Frequent Conjs. Come First',
 '5000 Most Common French Words::[A. 1] Irregular Verbs Training::[a] Option 2: One Verb a

In [ ]:
invoke('')

In [10]:
invoke('createDeck', deck='test1')

1741653962153

In [11]:
# invoke('deleteDecks', decks="test2", cardsToo=True)

In [12]:
# invoke('modelNames')

In [13]:
# print my vocab mining card type
invoke("modelTemplates", modelName="Vocab Mining")

{'Card 1': {'Front': '<span style="font-size: 50px;">{{Target Word}}</span><br>\n{{Word Audio}}',
  'Back': '{{FrontSide}}\n<hr id=answer>\n<span style="font-size: 35px;">{{Source Translation}}</span><br>\n{{Word Audio}}{{Sentence Audio}}<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'},
 'Card 2': {'Front': '<span style="font-size: 50px;">{{Source Translation}}</span>',
  'Back': '{{FrontSide}}\n\n<hr id=answer>\n<span style="font-size: 35px;">{{Target Word}}</span><br>\n{{Word Audio}}{{Sentence Audio}}<br>\n<span style="font-size: 25px;">{{Target Sentence}}<br></span>\n{{Source Sentence Translation}}<br>\n'}}

In [14]:
invoke("modelFieldNames", modelName="Vocab Mining")

['Target Word',
 'Source Translation',
 'Target Sentence',
 'Source Sentence Translation',
 'Word Audio',
 'Sentence Audio']

In [16]:
# generate test fields
"""
{
    "vocab_word": vocab_word,
    "vocab_translation": vocab_translation,
    "example_sentence": example_sentence,
    "example_sentence_translation": example_sentence_translation,
    "vocab_audio": vocab_audio,
    "example_sentence_translation_audio": example_sentence_audio
}
"""
# todo: infer language based on input?
result = process_vocab_word("制す", target_language="Japanese")


Processing vocabulary word: 制す

Checking if audio files exist...
制す_word.mp3: True
制す_sentence.mp3: True


In [17]:
result

{'vocab_word': '制す',
 'vocab_translation': 'Control',
 'example_sentence': '彼は自己の感情を制すことができました。',
 'example_sentence_translation': 'He was able to control his emotions.',
 'vocab_audio_filename': '制す_word.mp3',
 'example_sentence_translation_audio_filename': '制す_sentence.mp3'}

In [18]:
vocab_word = result["vocab_word"]
vocab_translation = result["vocab_translation"]
example_sentence = result["example_sentence"]
example_sentence_translation = result["example_sentence_translation"]
vocab_audio_filename = result["vocab_audio_filename"]
example_sentence_translation_audio_filename = result["example_sentence_translation_audio_filename"]

In [19]:
store_audio_file(vocab_audio_filename)
store_audio_file(example_sentence_translation_audio_filename)

/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_word.mp3
Stored 制す_word.mp3: 制す_word.mp3
/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_sentence.mp3
Stored 制す_sentence.mp3: 制す_sentence.mp3


In [20]:
vocab_audio_filename

'制す_word.mp3'

In [21]:
import os

print("Checking if audio files exist...")
print(f"{vocab_audio_filename}: {os.path.exists(vocab_audio_filename)}")
print(f"{example_sentence_translation_audio_filename}: {os.path.exists(example_sentence_translation_audio_filename)}")

Checking if audio files exist...
制す_word.mp3: True
制す_sentence.mp3: True


In [22]:

deck_name = f"test1"
model_name = "Vocab Mining"
fields = {
    'Target Word': vocab_word,
    'Source Translation': vocab_translation,
    'Target Sentence': example_sentence,
    'Source Sentence Translation':example_sentence_translation,
}
audio = [
    {
        "filename": vocab_audio_filename,
        "path": os.path.abspath(vocab_audio_filename),
        "fields": [
            "Word Audio"
        ]
    },
    {
        "filename": example_sentence_translation_audio_filename,
        "path": os.path.abspath(example_sentence_translation_audio_filename),
        "fields": [
            "Sentence Audio"
        ]
    }
]
note = {
    "deckName": deck_name,
    "modelName": model_name,
    "fields": fields,
    "audio": audio,
    "tags": ["stenchtoast"],
    "options": {
            "allowDuplicate": False,
            "duplicateScope": "deck",
            "duplicateScopeOptions": {
                "deckName": deck_name,
                "checkChildren": False,
                "checkAllModels": False
            }
    }
}

In [23]:
note

{'deckName': 'test1',
 'modelName': 'Vocab Mining',
 'fields': {'Target Word': '制す',
  'Source Translation': 'Control',
  'Target Sentence': '彼は自己の感情を制すことができました。',
  'Source Sentence Translation': 'He was able to control his emotions.'},
 'audio': [{'filename': '制す_word.mp3',
   'path': '/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_word.mp3',
   'fields': ['Word Audio']},
  {'filename': '制す_sentence.mp3',
   'path': '/Users/sethdonaldson/sourcecode/anki-vocab-generator/notebook/制す_sentence.mp3',
   'fields': ['Sentence Audio']}],
 'tags': ['stenchtoast'],
 'options': {'allowDuplicate': False,
  'duplicateScope': 'deck',
  'duplicateScopeOptions': {'deckName': 'test1',
   'checkChildren': False,
   'checkAllModels': False}}}

In [24]:
# add card to deck
invoke("addNote", note=note)

1742759997575